## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_10_ExG.csv'
MARKER_FILE = 'Case_10_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
# ica.plot_properties(raw, picks=range(15))

In [ ]:
exclude_indices = [0, 1, 4, 6, 7, 9]

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = "F3" if "F3" in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    raw.get_data(picks=target_chan)[0, 1000:2500],
    color="red",
    alpha=0.5,
    label="Before cleaning (Original)",
)
ax.plot(
    raw_clean.get_data(picks=target_chan)[0, 1000:2500],
    color="black",
    label="After cleaning (Clean)",
)
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz): Asymetria i Sterowanie
Widać tu przykład dysocjacji. Cała lewa półkula (C3, P3, O1) jest wyciszona/niebieska (co w paśmie Alpha oznacza silną aktywność kory). Cała prawa półkula (C4, P4, O2) płonie na czerwono (stan głębokiego spoczynku/transu).

Mechanika i Poznanie: Mózg perfekcyjnie izoluje prawą rękę do obsługi telefonu (niebieskie C3 vs czerwone C4).

Ponieważ lewa kora potyliczno-ciemieniowa przetwarza język i detale, a prawa ogarnia obraz całościowo, ten pacjent nie patrzy na filmy jako na całość. On analitycznie "czyta" ekran (napisy, poszczególne piksele, detale), całkowicie wyłączając prawopółkulowe chłonięcie klimatu czy muzyki. Niezależnie od fazy.

#### Analiza Pasma Beta (13–30 Hz)
Obserwacja potylicy (O1): Lewa kora wzrokowa (O1) jest trwale aktywna we wszystkich trybach. Potwierdza to, że uczestnik nieustannie filtruje i skanuje drobne informacje.

Anomalia Prawego Czoła (F4/F6): W fazach biernego oglądania (Brainrot) prawa kora czołowa jest lekko wybudzona.

W fazie Smart Scroll pojawia się tam pożar (ciemnoczerwona plama). Wymagające treści analityczne połączone z koniecznością nawigacji wymuszają na prawym płacie czołowym wysiłek związany z uwagą i orientacją przestrzenną.

#### Analiza Pasma Delta (1–4 Hz)
U tego badanego Delta nie rozlewa się na cały mózg (nie zasypia on w całości). Delta skupia się w jednym, potężnym ognisku nad prawym czołem (F4/F6).

**Smart (bez scrolla)**: Całkowity brak Delty (mózg w optymalnym stanie analitycznym).

**Brainrot Scroll**: Pojawia się czerwona plama zmęczenia.

**Smart Scroll**: Najgłębsza czerwień i najwyższy poziom Delty.

U badanego to nie treść go męczy, lecz sam interfejs i wymóg interakcji. Samo przewijanie ekranu (szczególnie, gdy musi skupić się na mądrych treściach) wywołuje u niego ostre, punktowe przeciążenie (tzw. znużenie poznawcze) prawego płata czołowego.

#### Analiza Pasma Gamma (>30 Hz): Koszt Fizjologiczny
**Brainrot vs Smart Scroll**: * W fazie Brainrot widać duży koszt fizjologiczny (skok Gammy) w rejonach skroniowo-ciemieniowych, co odpowiada za próbę zintegrowania szumu dźwiękowego z szybkimi obrazami.

Jednak absolutnym ekstremum jest faza Smart Scroll. Wystrzał Gammy w prawym płacie czołowym pokrywa się z pożarem z pasma Delta i Beta.

Próba analizy mądrych tekstów (lewa półkula) przy jednoczesnym fizycznym nawigowaniu po interfejsie (prawa kora czołowa) doprowadza procesor tego badanego do sporego wysiłku.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

#### Kora Czołowa (F4 vs F3)
Słupki FAA: Pokazują wysoki plus.

Topomapy: Pokazują gigantyczną falę Delta (znużenie) i Alfa (wyłączenie) dokładnie nad prawym czołem.

Badany nie odczuwa przyjemności ani motywacji dążeniowej. Interfejs smartfona i wymóg ciągłego podejmowania decyzji (scrollowania) tak mocno przebodźcowują jego prawy płat czołowy (odpowiedzialny za orientację, hamowanie impulsów i radzenie sobie z nowością), że ulega on punktowemu, ostremu znużeniu poznawczemu (Delta). Z powodu tego "zacięcia się" prawej półkuli, lewa strona przejmuje stery. W rzeczywistości badany działa na autopilocie, wyłączając emocjonalną/przestrzenną kontrolę.

#### Kora Potyliczna (O2 vs O1)
Wskaźnik na słupkach jest głęboko na minusie, a topomapy pokazują, że lewa kora wzrokowa odpoczywa (niebieska), a prawa pracuje (niższa Alfa).

Badany próbuje przetwarzać obrazy i układ aplikacji całościowo, przestrzennie (prawą półkulą). Niestety, ponieważ jego prawo-półkulowy układ dowodzenia (czoło) "leży" z przeciążenia układem interfejsu, to całościowe patrzenie staje się bezowocnym wysiłkiem. Próbuje chłonąć obrazki, ale interfejs zmusza go do czytania i logiki.

#### Kora Ruchowa (C4 vs C3)
Słupki centralne i topomapy pokazują izolację prawej ręki.

Badany przewija mechanicznie. Z powodu awarii prawego czoła, jego mózg ratuje się ucieczką w rutynę. Lewa półkula (sterująca prawym palcem) wykonuje po prostu powtarzalny, zautomatyzowany ruch scrollowania. To ratunek układu nerwowego przed całkowitym paraliżem analitycznym.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_10_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora wzrokowa (Alpha O1)

**Brainrot Scroll**: Fale osiągają wysokie, niefizjologiczne wartości rzędu 800 µV. W normalnym EEG to się nie zdarza. Może to oznaczać, że badany w trakcie oglądania szybkiego "Brainrotu" najprawdopodobniej skrajnie mrużył oczy z wysiłku, zaciskał zęby lub jego gałki oczne wykonywały gwałtowne, spazmatyczne wręcz ruchy (oczopląs), próbując nadążyć za chaosem na ekranie.

**Wpływ scrolla**: Za każdym razem, gdy pojawia się czerwona linia (scroll), gigantyczny szum uderza w samo dno (blisko zera). Ruch palcem to dla kory wzrokowej awaryjny "reset" – desperacka próba wymazania poprzedniego przebodźcowującego obrazu.

**Smart Scroll**: Skala wraca do normalnych wartości. Oznacza to, że przy statycznym tekście jego oczy i twarz uspokajają się fizycznie, a kora wzrokowa próbuje pracować "normalnie".

#### Kora ruchowa (Beta C3)
Na wykresie C3 (zarówno dla Brainrot, jak i Smart) nie ma zjawiska ERD/ERS (czyli delikatnego spadku fali przed ruchem i ładnego wyskoku po nim).

Zamiast tego linia trendu Bety to jedna wielka, ciągła, poszarpana "szczotka" o stałej mocy. Układ ruchowy nie "planuje" każdego przewinięcia. Ręka jest w ciągłym, izometrycznym napięciu, a kciuk przewija ekran odruchowo.

#### Płaty czołowe (Theta F4 i Fz)
Brainrot i Smart (wykresy F4): U zdrowego, zaangażowanego odbiorcy po każdym niebieskim/czerwonym scrollu widać było ładną "górkę" fali Theta (moment skupienia i zrozumienia nowej treści).

U tego badanego nie ma żadnego cyklu zrozumienia. Wykres F4 po prostu jednostajnie drży i skacze w sposób całkowicie losowy względem linii scrollowania. Płat czołowy jest "zawieszony". Zrobienie scrolla nie wywołuje skupienia na nowej treści, bo "procesor" nie jest w stanie przanalizować nic nowego.

#### Płaty ciemieniowe (Theta P4)
Kora ciemieniowa odpowiada za nawigację i odnalezienie się w przestrzeni interfejsu.

Wykresy dla P4. Fale Theta osiągają tam bardzo wysokie wartości.

Prawa kora ciemieniowa dosłownie "gotuje się" z wysiłku. Badany, mając wyłączony płat czołowy i przestymulowany wzrok, próbuje resztkami sił zorientować się w strukturze aplikacji.

--- 
Dla tego badanego scroll nie jest narzędziem do eksploracji. Scroll jest u niego mechanicznym tikiem (szum na C3), który na ułamek sekundy przynosi ulgę przebodźcowanym oczom, ale nie prowadzi absolutnie do żadnego nowego zrozumienia ani zaangażowania czołowego.